# 01 · Preprocessing (Adapted)

Adapted preprocessing pipeline for the Portuguese Public Procurement dataset.
Source: base.gov.pt · Contracts 2009-2022.

**Steps:**
1. Load raw contracts data
2. Filter & clean records
3. Restrict to open-tender (competitive) procedures
4. Identify the active-core subset (≈3 000 firms / 85 % expenditure)
5. Export cleaned dataset

In [ ]:
import pandas as pd
import numpy as np
import os

# ── Paths ──────────────────────────────────────────────────────────────
DATA_DIR  = os.path.join('data', 'raw')
OUT_DIR   = os.path.join('data', 'processed')
os.makedirs(OUT_DIR, exist_ok=True)

print('Directories ready.')
print(f'  raw      → {DATA_DIR}')
print(f'  processed → {OUT_DIR}')


## 1 · Load raw data

In [ ]:
# Load the BASE.gov.pt contracts CSV (adjust filename as needed)
# df_raw = pd.read_csv(os.path.join(DATA_DIR, 'contracts_raw.csv'),
#                      parse_dates=['signing_date', 'execution_date'],
#                      low_memory=False)

# --- DEMO: synthetic skeleton so the notebook runs stand-alone ---
np.random.seed(42)
N = 500
df_raw = pd.DataFrame({
    'contract_id'    : range(N),
    'firm_id'        : np.random.randint(1, 200, N),
    'entity_id'      : np.random.randint(1,  50, N),
    'procedure_type' : np.random.choice(['open_tender','direct_award','prior_consultation'], N),
    'contract_value' : np.random.exponential(50_000, N),
    'cpv_code'       : np.random.choice(['45000000','33000000','72000000','50000000'], N),
    'nuts2_region'   : np.random.choice(['PT11','PT15','PT16','PT17','PT18','PT20','PT30'], N),
    'signing_date'   : pd.date_range('2011-01-01', periods=N, freq='D')[:N]
})
print(f'Raw records: {len(df_raw):,}')
df_raw.head()


## 2 · Basic cleaning

In [ ]:
# Remove negative / zero contract values
df = df_raw[df_raw['contract_value'] > 0].copy()

# Drop rows with missing firm / entity / date
df.dropna(subset=['firm_id', 'entity_id', 'signing_date'], inplace=True)

# Restrict to 2011-2022 (three complete legislatures)
df = df[(df['signing_date'].dt.year >= 2011) &
        (df['signing_date'].dt.year <= 2022)]

print(f'After cleaning: {len(df):,} contracts')


## 3 · Filter to open tenders only

In [ ]:
df_tenders = df[df['procedure_type'] == 'open_tender'].copy()
print(f'Open-tender contracts: {len(df_tenders):,}')
print(f'Unique firms         : {df_tenders["firm_id"].nunique():,}')
print(f'Unique entities      : {df_tenders["entity_id"].nunique():,}')


## 4 · Active-core subset (≈85 % of expenditure)

In [ ]:
# Rank firms by total procurement revenue
firm_revenue = (df_tenders
                .groupby('firm_id')['contract_value']
                .sum()
                .sort_values(ascending=False))

total_rev   = firm_revenue.sum()
cum_share   = firm_revenue.cumsum() / total_rev
active_core = cum_share[cum_share <= 0.85].index

df_core = df_tenders[df_tenders['firm_id'].isin(active_core)].copy()

print(f'Active-core firms : {len(active_core):,}  '
      f'({100*len(active_core)/df_tenders["firm_id"].nunique():.1f} % of all firms)')
print(f'Expenditure share : '
      f'{100*df_core["contract_value"].sum()/df_tenders["contract_value"].sum():.1f} %')


## 5 · Export

In [ ]:
out_path = os.path.join(OUT_DIR, 'contracts_core.parquet')
# df_core.to_parquet(out_path, index=False)
# print(f'Saved → {out_path}')
print('Export step ready (uncomment to write parquet).')
